In [1]:
%pip install -q -U transformers accelerate peft trl datasets bitsandbytes huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [4]:
!nvidia-smi

Mon Jul 20 01:38:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 576.28                 Driver Version: 576.28         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   38C    P0             11W /   65W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Uncomment and adjust the CUDA version tag (cu121 / cu124 / cu128 ...) to match your driver, then run.
%pip uninstall torch torchvision torchaudio -y
%pip install torch --index-url https://download.pytorch.org/whl/cu124

import torch
print("Torch version:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())


Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
You can safely remove it manually.


Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached torch-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl.metadata (28 kB)
Using cached torch-2.6.0%2Bcu124-cp311-cp311-win_amd64.whl (2532.4 MB)
Note: you may need to restart the kernel to use updated packages.
Torch version: 2.6.0+cu124
Torch CUDA build: 12.4
CUDA available: True


In [6]:
import os
import json
import random

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig

random.seed(42)
torch.manual_seed(42)


In [7]:
if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"CUDA available: True")
    print(f"GPU: {gpu_name}")
    print(f"Total VRAM: {total_mem_gb:.1f} GB")
    # bf16 is supported on Ampere+ (compute capability >= 8.0)
    major, minor = torch.cuda.get_device_capability(0)
    USE_BF16 = major >= 8
    print(f"Compute capability: {major}.{minor} -> using {'bf16' if USE_BF16 else 'fp16'}")
else:
    device = "cpu"
    USE_BF16 = False
    print("CUDA not available -- falling back to CPU. Training will be slow.")
    print("If you have an NVIDIA GPU, see the CUDA-fix cell above (section 1b).")

print(f"\nSelected device: {device}")


CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM: 6.0 GB
Compute capability: 8.9 -> using bf16

Selected device: cuda


In [8]:
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Use 4-bit quantization if on GPU to keep memory low; full precision on CPU.
if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float32,
    ).to(device)

print(base_model.config)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  

In [ ]:
DATA_DIR = "."

train_path = os.path.join(DATA_DIR, "train.jsonl")
val_path = os.path.join(DATA_DIR, "val.jsonl")

raw_datasets = load_dataset(
    "json",
    data_files={"train": train_path, "validation": val_path},
)

print(raw_datasets)
print("\nExample raw record:")
print(json.dumps(raw_datasets["train"][0], indent=2)[:1000])


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 950
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 50
    })
})

Example raw record:
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert Information Extraction AI. Parse messy OCR input into a valid JSON matching the requested schema. Correct OCR typos, missing spaces, and noise."
    },
    {
      "role": "user",
      "content": "Extract fields into JSON: {\"vendor\": string, \"invoice_no\": string, \"date\": string, \"total\": float, \"items\": [{ \"desc\": string, \"amount\": float }]}\n\nRAW OCR INPUT:\nINVOICE NO: 275022449l  DATE: 20250826 VENDOR: Sunshine Bakery Supplies\nITEMS:\n1. Circuit Breaker 32A - $35,072.44 _\n2. LED Panel Light 18W - $20,306.30\n3. Conveyor Belt Roll - $16,o85.78\n4. TMT Bar 12mm - $23,95Z.36\nTOTAL: $95,4I6.88"
    },
    {
      "role": "assistant",
      "content": "```json\n{\n  \"vendor\": \"Suns

In [10]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_train = raw_datasets["train"].map(format_example, remove_columns=raw_datasets["train"].column_names)
formatted_val = raw_datasets["validation"].map(format_example, remove_columns=raw_datasets["validation"].column_names)

print("Formatted example:\n")
print(formatted_train[0]["text"])


Map:   0%|          | 0/950 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Formatted example:

<|im_start|>system
You are an expert Information Extraction AI. Parse messy OCR input into a valid JSON matching the requested schema. Correct OCR typos, missing spaces, and noise.<|im_end|>
<|im_start|>user
Extract fields into JSON: {"vendor": string, "invoice_no": string, "date": string, "total": float, "items": [{ "desc": string, "amount": float }]}

RAW OCR INPUT:
INVOICE NO: 275022449l  DATE: 20250826 VENDOR: Sunshine Bakery Supplies
ITEMS:
1. Circuit Breaker 32A - $35,072.44 _
2. LED Panel Light 18W - $20,306.30
3. Conveyor Belt Roll - $16,o85.78
4. TMT Bar 12mm - $23,95Z.36
TOTAL: $95,4I6.88<|im_end|>
<|im_start|>assistant
```json
{
  "vendor": "Sunshine Bakery Supplies",
  "invoice_no": "2750224491",
  "date": "2025-08-26",
  "total": 95416.88,
  "items": [
    {
      "desc": "Circuit Breaker 32A",
      "amount": 35072.44
    },
    {
      "desc": "LED Panel Light 18W",
      "amount": 20306.3
    },
    {
      "desc": "Conveyor Belt Roll",
      "amount

In [11]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [15]:
OUTPUT_DIR = "./qwen25-1_5b-ocr-json-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=USE_BF16 and device == "cuda",
    fp16=(not USE_BF16) and device == "cuda",
    report_to="none",
    dataset_text_field="text",
    packing=False,
    max_length=1024,
)

trainer = SFTTrainer(
    model=peft_model,
    args=sft_config,
    train_dataset=formatted_train,
    eval_dataset=formatted_val,
    processing_class=tokenizer,
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/950 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

In [16]:
trainer.train()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.323528,0.298809,0.298260,288780.000000,0.903184
100,0.275737,0.274011,0.270165,572094.000000,0.906199
150,0.276854,0.269211,0.262897,857143.000000,0.906885
180,0.272807,0.269178,0.263717,1027335.000000,0.907226


TrainOutput(global_step=180, training_loss=0.3746347943941752, metrics={'train_runtime': 1238.7845, 'train_samples_per_second': 2.301, 'train_steps_per_second': 0.145, 'total_flos': 9113760497498112.0, 'train_loss': 0.3746347943941752, 'epoch': 3.0})

In [17]:
FINAL_ADAPTER_DIR = "./qwen25-1_5b-ocr-json-lora/final_adapter"
trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Saved LoRA adapter to {FINAL_ADAPTER_DIR}")


Saved LoRA adapter to ./qwen25-1_5b-ocr-json-lora/final_adapter


In [18]:
TEST_PROMPT_SYSTEM = (
    "You are an expert Information Extraction AI. Parse messy OCR input into "
    "a valid JSON matching the requested schema. Correct OCR typos, missing "
    "spaces, and noise."
)

TEST_SCHEMA = (
    '{"vendor": string, "invoice_no": string, "date": string, "total": float, '
    '"items": [{ "desc": string, "amount": float }]}'
)

TEST_RAW_OCR = """INV0ICE N0: #7734-C  DATE: 2O26/03/15 VEND0R: Zenith Manufacturing
ITEM S: 1. Hex B0lt M10 - $1,2SO.OO 2. Grinding Wheel 4inch - 3,410.5O
T0TAL: 4,660.50"""

test_messages = [
    {"role": "system", "content": TEST_PROMPT_SYSTEM},
    {
        "role": "user",
        "content": f"Extract fields into JSON: {TEST_SCHEMA}\n\nRAW OCR INPUT:\n{TEST_RAW_OCR}",
    },
]

def generate_response(model, tokenizer, messages, max_new_tokens=300):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


In [ ]:
print("Loading fresh untrained base model for comparison...")

if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    untrained_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
    )
else:
    untrained_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float32,
    ).to(device)

untrained_model.eval()

print("\n" + "=" * 70)
print("BASE MODEL (untrained) OUTPUT")
print("=" * 70)
base_output = generate_response(untrained_model, tokenizer, test_messages)
print(base_output)


Loading fresh untrained base model for comparison...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


BASE MODEL (untrained) OUTPUT
```json
{
  "vendor": "Zenith Manufacturing",
  "invoice_no": "#7734-C",
  "date": "2026-03-15",
  "total": 4660.5,
  "items": [
    {
      "desc": "Hex Bolt M10",
      "amount": 1200.0
    },
    {
      "desc": "Grinding Wheel 4inch",
      "amount": 3410.5
    }
  ]
}
```


In [ ]:
print("Loading fine-tuned model (base + LoRA adapter)...")

finetuned_model = PeftModel.from_pretrained(untrained_model, FINAL_ADAPTER_DIR)
finetuned_model.eval()

print("\n" + "=" * 70)
print("FINE-TUNED MODEL OUTPUT")
print("=" * 70)
finetuned_output = generate_response(finetuned_model, tokenizer, test_messages)
print(finetuned_output)


Loading fine-tuned model (base + LoRA adapter)...

FINE-TUNED MODEL OUTPUT
```json
{
  "vendor": "Zenith Manufacturing",
  "invoice_no": "#7734-C",
  "date": "2026-03-15",
  "total": 4660.5,
  "items": [
    {
      "desc": "Hex Bolt M10",
      "amount": 1250.0
    },
    {
      "desc": "Grinding Wheel 4inch",
      "amount": 3410.5
    }
  ]
}
```


In [21]:
import re

def extract_json_block(text):
    match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    candidate = match.group(1) if match else text
    try:
        return json.loads(candidate), None
    except Exception as e:
        return None, str(e)

print("RAW OCR INPUT:\n")
print(TEST_RAW_OCR)

print("\n" + "-" * 70)
print("BASE MODEL OUTPUT:")
print("-" * 70)
print(base_output)
base_parsed, base_err = extract_json_block(base_output)
print(f"\nValid JSON? {'YES' if base_parsed else 'NO -- ' + str(base_err)}")

print("\n" + "-" * 70)
print("FINE-TUNED MODEL OUTPUT:")
print("-" * 70)
print(finetuned_output)
ft_parsed, ft_err = extract_json_block(finetuned_output)
print(f"\nValid JSON? {'YES' if ft_parsed else 'NO -- ' + str(ft_err)}")

print("\n" + "=" * 70)
print("EXPECTED (ground truth for this sample):")
print("=" * 70)
expected = {
    "vendor": "Zenith Manufacturing",
    "invoice_no": "#7734-C",
    "date": "2026-03-15",
    "total": 4660.50,
    "items": [
        {"desc": "Hex Bolt M10", "amount": 1250.00},
        {"desc": "Grinding Wheel 4inch", "amount": 3410.50},
    ],
}
print(json.dumps(expected, indent=2))


RAW OCR INPUT:

INV0ICE N0: #7734-C  DATE: 2O26/03/15 VEND0R: Zenith Manufacturing
ITEM S: 1. Hex B0lt M10 - $1,2SO.OO 2. Grinding Wheel 4inch - 3,410.5O
T0TAL: 4,660.50

----------------------------------------------------------------------
BASE MODEL OUTPUT:
----------------------------------------------------------------------
```json
{
  "vendor": "Zenith Manufacturing",
  "invoice_no": "#7734-C",
  "date": "2026-03-15",
  "total": 4660.5,
  "items": [
    {
      "desc": "Hex Bolt M10",
      "amount": 1200.0
    },
    {
      "desc": "Grinding Wheel 4inch",
      "amount": 3410.5
    }
  ]
}
```

Valid JSON? YES

----------------------------------------------------------------------
FINE-TUNED MODEL OUTPUT:
----------------------------------------------------------------------
```json
{
  "vendor": "Zenith Manufacturing",
  "invoice_no": "#7734-C",
  "date": "2026-03-15",
  "total": 4660.5,
  "items": [
    {
      "desc": "Hex Bolt M10",
      "amount": 1250.0
    },
    {
    